# Regression

> The objective of this notebook is to create a simple scikit-learn regression wrapper that will ease model training and benchmarking

In [ ]:
#| default_exp classicml.regression

In [4]:
#| hide
from nbdev.showdoc import *
import numpy as np

/home/i/Documentos/1_Proyectos/omvs_senegal/.venv/lib/python3.12/site-packages/nbdev/doclinks.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources,importlib


In [1]:
#| export
import pandas as pd
from sklearn.linear_model import LinearRegression

In [2]:
#| export
class SimpleRegressionModel:
    def __init__(self):
        self.model = LinearRegression()
    
    def fit(self, X, y):
        self.model.fit(X, y)
        return self
    
    def predict(self, X):
        return self.model.predict(X)
    
    def predict_as_dataframe(self, X, degree, context_len):
        pred = self.predict(X)
        pred = pd.DataFrame(pred, index=X.index, columns=[f"t+{i+1}" for i in range(0, pred.shape[1])])
        pred.columns.name = "forecast_horizon"
        pred.index.name = "run_time"
        pred['degree'] = degree
        pred['context_len'] = context_len
        pred = pred.set_index(['degree', 'context_len'], append=True)
        pred = pred.reorder_levels(["degree", "context_len", "run_time"]).sort_index()
        return pred

In [5]:
#| hide
def create_synthetic_data():
    """Create synthetic data for testing"""
    run_times = pd.date_range(start='2023-01-01', end='2023-12-31', freq='D', name="run_time")  
    horizons = [f't+{i}' for i in range(1, 4)] # Forecast horizons (t+1, t+2, t+3)
    data_y = pd.DataFrame(index=run_times, columns=horizons, data=np.random.rand(len(run_times), len(horizons)) * 100)
    
    time_index = pd.date_range(start='2023-01-01', end='2023-12-31', freq='D', name="time")  
    vars = [f'Var {i}' for i in range(1, 5)] # Forecast horizons (t+1, t+2, t+3)
    data_x = pd.DataFrame(index=time_index, columns=vars, data=np.random.rand(len(time_index), len(vars)) * 100)
    
    
    return data_x, data_y

We can define and train a simple model as follows

In [ ]:
model = SimpleRegressionModel()
model.fit(train_x, train_y)

<__main__.SimpleRegressionModel>

And we can get the raw prediction

In [ ]:
model.predict(valid_x)[:3]

array([[66.40067038, 64.0534088 ],
       [65.55067375, 63.23207101],
       [64.71676498, 62.42401873]])

Or the prediction with indexes as follows

In [ ]:
pred = model.predict_as_dataframe(valid_x, degree=2, context_len=1)
pred.head(3)

forecast_horizon                     t+1        t+2
degree context_len run_time                        
2      1           2019-01-01  66.400670  64.053409
                   2019-01-02  65.550674  63.232071
                   2019-01-03  64.716765  62.424019